**실습 목표**

- 텍스트 분류 데이터의 빈 문장, label 오류, 중복을 점검합니다.
- Class 분포와 imbalance ratio를 계산합니다.
- Train·Validation 사이의 데이터 leakage를 찾습니다.

---

## 핵심 실습. 텍스트 분류 데이터 품질 점검

### 시작 코드

```python
records = [
    {"id": "A1", "text": "접근 권한을 확인해주세요", "label": "보안"},
    {"id": "A2", "text": "  ", "label": "일반"},
    {"id": "A3", "text": "서버가 응답하지 않습니다", "label": "장애"},
    {"id": "A4", "text": "접근   권한을 확인해주세요", "label": "보안"},
    {"id": "A5", "text": None, "label": "기타"},
]
allowed_labels = {"일반", "보안", "장애"}
```

### 수행해야 할 작업

1. 공백을 하나로 합치고 앞뒤 공백을 제거하는 `normalize_text`를 작성하세요.
2. 빈 문장, 문자열이 아닌 문장, 허용되지 않은 label을 찾으세요.
3. 정규화 후 같은 문장의 중복 id를 찾으세요.
4. 정상 record와 오류 리포트를 분리하세요.
    
**해설**
    
 정규화하지 않고 문자열을 그대로 비교하면 공백 차이로 같은 문장을 놓칠 수 있습니다. 중복 제거 전에는 같은 문장이 실제 반복 관측인지 수집 오류인지 도메인 기준으로 확인하세요.

In [1]:
records = [
    {"id":"A1","text":"접근 권한을 확인해주세요","label":"보안"},
    {"id":"A2","text":"  ","label":"일반"},
    {"id":"A3","text":"서버가 응답하지 않습니다","label":"장애"},
    {"id":"A4","text":"접근   권한을 확인해주세요","label":"보안"},
    {"id":"A5","text":None,"label":"기타"},
]
allowed_labels = {"일반", "보안", "장애"}


def normalize_text(text):
    return " ".join(text.split())


def audit_classification_records(records, allowed_labels):
    valid, errors = [], []
    first_seen = {}

    for record in records:
        text = record.get("text")
        if not isinstance(text, str) or not text.strip():
            errors.append({"id": record.get("id"), "reason": "invalid_text"})
            continue
        if record.get("label") not in allowed_labels:
            errors.append({"id": record.get("id"), "reason": "invalid_label"})
            continue

        normalized = normalize_text(text)
        if normalized in first_seen:
            errors.append({"id": record["id"], "reason": "duplicate_text", "duplicate_of": first_seen[normalized]})
            continue

        first_seen[normalized] = record["id"]
        valid.append({**record, "text": normalized})

    return {"valid_records": valid, "errors": errors}


report = audit_classification_records(records, allowed_labels)
print(report)
assert [row["id"] for row in report["valid_records"]] == ["A1", "A3"]
assert [row["reason"] for row in report["errors"]] == ["invalid_text", "duplicate_text", "invalid_text"]

{'valid_records': [{'id': 'A1', 'text': '접근 권한을 확인해주세요', 'label': '보안'}, {'id': 'A3', 'text': '서버가 응답하지 않습니다', 'label': '장애'}], 'errors': [{'id': 'A2', 'reason': 'invalid_text'}, {'id': 'A4', 'reason': 'duplicate_text', 'duplicate_of': 'A1'}, {'id': 'A5', 'reason': 'invalid_text'}]}


## 핵심 보조 실습. Label 분포와 Imbalance 계산

### 시작 코드

```python
labels = ["일반", "일반", "일반", "일반", "보안", "보안", "장애"]
```

### 수행해야 할 작업

1. Label별 개수와 비율을 계산하세요.
2. 최대 class 수 / 최소 class 수를 imbalance ratio로 계산하세요.
3. 비율이 10% 미만인 minority class를 표시하세요.
4. Accuracy만 사용할 때 생길 수 있는 문제를 한 문장으로 설명하세요.
  
  **해설**
    
  소수 class를 거의 맞히지 못해도 다수 class 비중이 크면 Accuracy가 높게 나올 수 있습니다. 불균형 분류에서는 class별 recall과 Macro-F1을 함께 확인하세요.

In [5]:
from collections import Counter

labels = ["일반","일반","일반","일반","보안","보안","장애"]


def summarize_label_distribution(labels, minority_threshold=0.1):
    if not labels:
        raise ValueError("label 목록이 비어 있습니다.")

    counts = Counter(labels)
    total = len(labels)
    ratios = {label: count / total for label, count in sorted(counts.items())}
    imbalance_ratio = max(counts.values()) / min(counts.values())

    return {
        "counts": dict(counts),
        "ratios": {k: round(v, 4) for k, v in ratios.items()},
        "imbalance_ratio": round(imbalance_ratio, 4),
        "minority_labels": [label for label, ratio in ratios.items() if ratio < minority_threshold],
    }


report = summarize_label_distribution(labels)
print(report)
assert report["counts"] == {"일반": 4, "보안": 2, "장애": 1}
assert report["imbalance_ratio"] == 4.0


{'counts': {'일반': 4, '보안': 2, '장애': 1}, 'ratios': {'보안': 0.2857, '일반': 0.5714, '장애': 0.1429}, 'imbalance_ratio': 4.0, 'minority_labels': []}


## 참고·심화 실습. Split Leakage 탐지

### 시작 코드

```python
train = [
    {"id": "T1", "text": "접근 권한 오류"},
    {"id": "T2", "text": "서버 연결 실패"},
    {"id": "T3", "text": "결제 문의"},
]
validation = [
    {"id": "V1", "text": "접근   권한 오류"},
    {"id": "V2", "text": "새로운 보안 문의"},
    {"id": "V3", "text": "서버 연결 실패!"},
]
```

### 수행해야 할 작업

1. 소문자, 공백 정리, 끝 문장부호 제거를 포함한 정규화 함수를 작성하세요.
2. Train과 Validation의 정확 중복을 찾으세요.
3. 단어 집합 Jaccard similarity가 0.8 이상인 near-duplicate도 찾으세요.
4. 중복 쌍의 id와 similarity를 반환하세요.
    
   **해설**
    
  Validation에 Train 문장이 섞이면 일반화 성능이 아니라 기억한 문장 재현 능력을 측정하게 됩니다. Split 이후가 아니라 Split 전에 중복 그룹을 묶어 같은 그룹이 한쪽에만 가도록 나누는 방법도 고려하세요.

In [6]:
import re

train = [{"id":"T1","text":"접근 권한 오류"},{"id":"T2","text":"서버 연결 실패"},{"id":"T3","text":"결제 문의"}]
validation = [{"id":"V1","text":"접근   권한 오류"},{"id":"V2","text":"새로운 보안 문의"},{"id":"V3","text":"서버 연결 실패!"}]


def normalize_for_leakage(text):
    text = re.sub(r"[.!?]+$", "", text.lower().strip())
    return " ".join(text.split())


def jaccard_words(left, right):
    left_set, right_set = set(left.split()), set(right.split())
    return len(left_set & right_set) / len(left_set | right_set) if left_set | right_set else 1.0


def find_split_leakage(train, validation, threshold=0.8):
    pairs = []
    for train_row in train:
        left = normalize_for_leakage(train_row["text"])
        for valid_row in validation:
            right = normalize_for_leakage(valid_row["text"])
            similarity = jaccard_words(left, right)
            if similarity >= threshold:
                pairs.append({
                    "train_id": train_row["id"],
                    "validation_id": valid_row["id"],
                    "similarity": round(similarity, 4),
                    "exact_after_normalization": left == right,
                })
    return pairs


leakage = find_split_leakage(train, validation)
print(leakage)
assert {(x["train_id"], x["validation_id"]) for x in leakage} == {("T1", "V1"), ("T2", "V3")}

[{'train_id': 'T1', 'validation_id': 'V1', 'similarity': 1.0, 'exact_after_normalization': True}, {'train_id': 'T2', 'validation_id': 'V3', 'similarity': 1.0, 'exact_after_normalization': True}]
